# 1.Feature Engineering

### (REVIEW)

Let's go over the description of what I intend to do in my project from what I said in deliverable 1:

### 1.1 Description of the problem

I want to investigate the accuracy and efficiency of Convolutional Neural Networks (CNNs) for autoregressive text generation. For this project, I want to train and test different CNN models by changing their parameters and embedding dimensions to see how well they can generate text. I also want to experiment with different levels of text preprocessing to see how this affects the models performance and ability to generate coherent text.

I will then compare the results using metrics such as tokens per second, perplexity, model size and cross-entropy loss to evaluate how well the different models perform.

I am going to use the 'WikiText-103' dataset, which is raw text, This dataset comes from huggingface:

https://huggingface.co/datasets/Salesforce/wikitext?utm_source=chatgpt.com

This file 'wikitext-103-v1' is much bigger at size - 738.27 MB, my MacBook M1 chip would struggle to process so we'll use a runpod GPU

I understand the assigned says:

Feature Engineering (30 points)
 - Text representation techniques (TF-IDF, Word2Vec, FastText, BERT embeddings, etc.) (WE'RE GOING TO BUILD OUR OWN VECTOR EMBEDDING)
 - Justification for the chosen representation (WE WANT TO INVESTIGATE THE AUTO GENERATIVE ABILITY OF CNNs MODELS)

Model Selection and Training (50 points)
 - Comparison of different models (e.g., Naïve Bayes, SVMs, CNNs, LSTMs, Transformers) (WE WILL COMPARE CNN AGAINST CNN BUT ALSO AGAINST LLM)
 - Justification for the selected approach
 - Model training implementation

Evaluation and Results (45 points)
 - Performance metrics (accuracy, precision, recall, F1-score, etc.)
 - Comparison of model performance
 - Discussion of findings

In [19]:
import torch
from datasets import load_dataset

print("GPU:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    dataset = load_dataset("Salesforce/wikitext","wikitext-103-raw-v1")
else:
    dataset = load_dataset("Salesforce/wikitext","wikitext-2-raw-v1")
    print("Running on CPU, use smaller dataset.")

GPU: False
Running on CPU, use smaller dataset.


Ok my above code is saying that if we have a GPU attached then we use the bigger raw text dataset, this is because I have coded a lot of this on my MAC CPU which can't handle that much data, so we use the smaller dataset at first to check my code is correct.

After I will then attach it to my runpod GPU which has much more powerful compute.

In [20]:
train = dataset["train"]
test = dataset["test"]

In [21]:
import random

num = random.randint(0,len(train)-1)

print(train[num])

{'text': ''}


In [22]:
df_train = train.to_pandas()

We're going to combine all the sentences into one massive text file as I want to cut them into 500 tokens each, that way we feed the model with no filler tokens.

In [23]:
all_text = " ".join(df_train["text"].dropna().astype(str))

In [24]:
all_text[:2000]

'  = Valkyria Chronicles III = \n   Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n  The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for seri

In [25]:
import string
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

from torch.utils.data import TensorDataset, DataLoader

# ============== SET GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("We use:", torch.cuda.get_device_name(0) if device.type == "cuda" else "My MAC")

# ============== TOKENISE TEXT

title_tokens = all_text.lower().split()

# ============== REMOVE STANDALONE PUNCTUATION

title_tokens = [word for word in title_tokens if word not in string.punctuation] #I STILL WANT TO REMOVE PUNCTUATION

# ============== COLLECT VOCABULARY

VOCAB = set(title_tokens)

VOCAB.add("<UNK>") #THIS IS FOR OUR UNKNOWN WORDS

vocab_tokens = {word: i for i, word in enumerate(sorted(VOCAB))}

print("Vocabulary size:", len(vocab_tokens))

# ============== SPLIT INTO GROUPS OF 500 TOKENS

processed_text = []

for i in range(0, len(title_tokens), 500):

    token_group = title_tokens[i:i + 500]

    if len(token_group) == 500:
        processed_text.append(token_group)

# ============== CREATE NEW DATAFRAME

df = pd.DataFrame({"processed_text": processed_text})

We use: My MAC
Vocabulary size: 66620


THIS NUMBER WILL CHANGE WHEN RUNNING IT ON THE RUNPOD CPU AS IT WILL USE THE - wikitext-103-raw-v1

Ok so we have 66620 tokens (wikitext-2-raw-v1) (words that have been tokenized), we then need to embed each of these words as we will do below.

In [26]:
df.head()

,processed_text
0,"[valkyria, chronicles, iii, senjō, no, valkyri..."
1,"[a, top, @-@, down, perspective, of, the, batt..."
2,"[gallia, lose, face, in, the, war, while, at, ..."
3,"[true, sequel, while, valkyria, chronicles, ii..."
4,"[music, he, felt, that, he, needed, to, incorp..."


In [27]:
vocab_tokens

{"'a": 0,
 "'aaj": 1,
 "'aak": 2,
 "'aaknal": 3,
 "'académie": 4,
 "'action": 5,
 "'ad": 6,
 "'adanle": 7,
 "'addario": 8,
 "'aeback": 9,
 "'affaires": 10,
 "'africaine": 11,
 "'ah": 12,
 "'ai": 13,
 "'ak": 14,
 "'akov": 15,
 "'al": 16,
 "'albert": 17,
 "'ale": 18,
 "'alesandro": 19,
 "'alkan": 20,
 "'amore": 21,
 "'amour": 22,
 "'an": 23,
 "'anan": 24,
 "'andrade": 25,
 "'ani": 26,
 "'anjou": 27,
 "'anshan": 28,
 "'apollon": 29,
 "'aquin": 30,
 "'aragona": 31,
 "'architecture": 32,
 "'aretz": 33,
 "'ari": 34,
 "'art": 35,
 "'artus": 36,
 "'asiyot": 37,
 "'assomption": 38,
 "'at": 39,
 "'atun": 40,
 "'aubeterre": 41,
 "'aubigny": 42,
 "'aude": 43,
 "'auteroche": 44,
 "'automobile": 45,
 "'autrui": 46,
 "'avria": 47,
 "'awiil": 48,
 "'ayab": 49,
 "'azzam": 50,
 "'brien": 51,
 "'callaghan": 52,
 "'carmel": 53,
 "'casey": 54,
 "'chard": 55,
 "'clock": 56,
 "'connell": 57,
 "'connor": 58,
 "'cruz": 59,
 "'d": 60,
 "'daniel": 61,
 "'day": 62,
 "'dell": 63,
 "'domo": 64,
 "'donnell": 65,
 "'

In [28]:
import random

num = random.randint(0,len(df)-1)

print(df["processed_text"].iloc[num])

['single', 'lay', 'lady', 'lay', 'variety', 'wrote', 'dylan', 'is', 'definitely', 'doing', 'something', 'that', 'can', 'be', 'called', 'singing', 'somehow', 'he', 'has', 'managed', 'to', 'add', 'an', 'octave', 'to', 'his', 'range', 'dylan', 'and', 'cash', 'also', 'recorded', 'a', 'series', 'of', 'duets', 'but', 'only', 'their', 'recording', 'of', 'dylan', "'s", 'girl', 'from', 'the', 'north', 'country', 'was', 'used', 'on', 'the', 'album', 'in', 'may', '1969', 'dylan', 'appeared', 'on', 'the', 'first', 'episode', 'of', 'johnny', 'cash', "'s", 'television', 'show', 'duetting', 'with', 'cash', 'on', 'girl', 'from', 'the', 'north', 'country', 'i', 'threw', 'it', 'all', 'away', 'and', 'living', 'the', 'blues', 'dylan', 'next', 'traveled', 'to', 'england', 'to', 'top', 'the', 'bill', 'at', 'the', 'isle', 'of', 'wight', 'festival', 'on', 'august', '31', '1969', 'after', 'rejecting', 'overtures', 'to', 'appear', 'at', 'the', 'woodstock', 'festival', 'closer', 'to', 'his', 'home', '1970s', 'in

In [29]:
VOCAB

{'kensington',
 'commandants',
 'plex',
 'nests',
 '808s',
 'mating',
 'dropping',
 'tsong',
 'sumter',
 'holding',
 'suede',
 'pains',
 'jyothi',
 'schnellenberger',
 'uppingham',
 'lutzenkirchen',
 'wollaston',
 'frescos',
 'polyurethane',
 'rossi',
 'notifying',
 'mutate',
 'ritgen',
 'léger',
 'melanophrys',
 'sunningdale',
 'reconnoiter',
 'nubian',
 'quatrain',
 'risk',
 'ledger',
 'blabbermouth.net',
 'bbc',
 'tr',
 'hille',
 'earn',
 'groupies',
 'hight',
 'duvendeck',
 'hygiene',
 'elevating',
 '194c',
 'cabaret',
 'contaminating',
 'marketed',
 'aromas',
 'santhush',
 'amina',
 'diiie',
 'synodic',
 'stuffs',
 'dhoti',
 'wittig',
 'edgartown',
 'walrus',
 'boastful',
 'muller',
 'bureau',
 'hawks',
 'guts',
 'sultani',
 'unsightly',
 'partition',
 'vibrancy',
 'stanleyana',
 'cvr',
 'trile',
 'unfriendly',
 '10t',
 '872',
 'overhear',
 'transverses',
 'karyo',
 'philharmonia',
 '1121',
 'ゲーティア',
 'sharpsburg',
 '1205',
 'guantone',
 'mummiform',
 'exquisite',
 'modelling',
 '

In [30]:
# ============== CREATE TOKEN IDS

unk_id = vocab_tokens["<UNK>"]

df["token_ids"] = df["processed_text"].apply(lambda batch: [vocab_tokens.get(word, unk_id) for word in batch])

# ============== CREATE TENSOR

token_tensor = torch.tensor(df["token_ids"].tolist(), dtype=torch.long)

print("Tensor shape:", token_tensor.shape)

Tensor shape: torch.Size([3559, 500])


In [31]:
df.head()

,processed_text,token_ids
0,"[valkyria, chronicles, iii, senjō, no, valkyri...","[62472, 12861, 29491, 53096, 41262, 62472, 185..."
1,"[a, top, @-@, down, perspective, of, the, batt...","[2935, 59870, 2931, 18988, 44654, 42084, 59070..."
2,"[gallia, lose, face, in, the, war, while, at, ...","[24559, 35694, 22071, 29849, 59070, 63736, 642..."
3,"[true, sequel, while, valkyria, chronicles, ii...","[60742, 53175, 64284, 62472, 12861, 29488, 267..."
4,"[music, he, felt, that, he, needed, to, incorp...","[39978, 27488, 22605, 59063, 27488, 40685, 596..."


Here we see the 'token_ids', if we look left then to right we see that we have replaced out processed words to tokens, now again each of these numbers will be given a vector of n length which is the embedding.

In [32]:
import random

num = random.randint(0,len(df)-1)

print("Length: ",len(df["token_ids"].iloc[num]), " Tokens: ",df["token_ids"].iloc[num])

Length:  500  Tokens:  [28918, 4950, 18597, 48742, 63818, 39534, 35149, 59672, 55516, 13044, 29849, 35312, 25609, 37400, 5537, 59672, 28934, 29849, 2935, 45001, 23698, 10652, 29849, 48716, 35040, 59070, 19981, 64143, 35149, 59672, 39406, 30089, 37637, 42084, 13989, 59070, 6999, 42084, 2935, 25590, 63818, 51655, 59672, 44548, 34546, 59070, 18597, 48741, 59672, 19550, 29849, 59070, 29570, 42084, 59063, 25590, 10710, 30354, 59178, 29570, 59070, 25609, 34608, 59094, 14187, 56175, 4950, 59855, 42307, 2935, 45001, 23698, 59672, 59070, 19981, 2935, 45352, 42496, 41870, 59063, 63818, 66330, 66345, 51567, 66345, 63818, 31277, 4950, 50672, 47577, 4950, 59394, 23079, 23578, 2935, 25590, 59672, 30348, 58789, 56203, 4950, 49524, 6117, 64110, 6117, 43866, 51567, 5087, 35100, 59070, 5484, 10420, 53245, 6117, 18597, 30777, 29849, 59241, 63936, 19147, 4950, 60212, 47249, 2935, 62872, 17878, 62773, 23578, 30701, 29849, 59178, 56182, 31326, 63818, 8132, 44443, 15215, 13864, 13378, 59672, 59070, 25609, 49

Above is just see a whole sentence tokenized, each number represents a word/symbol/number.

In [37]:
print(token_tensor.shape)

torch.Size([3559, 500])


For (wikitext-2-raw-v1) we're getting - torch.Size([3559, 500]) - This means that we have 3558 rows with 500 tokens each, if we look below i have give each word an embedding of 200, that means we have 500 * 200 for each row we process.

In [39]:
len(df)

3559

In [35]:
embedding = nn.Embedding(num_embeddings=len(vocab_tokens),embedding_dim=200).to(device)

In [36]:
embedding

Embedding(66620, 200)

For (wikitext-2-raw-v1) we're getting Embedding(66620, 200), which means that each token get's a 200 length vector embedding.

In [38]:
print(len(VOCAB))

66620


## IMPORTANT

Let's try something very cool, let's pick a selection of words that should relate to each other, say like fruit, we will check the distance between these words before and after to see of the model has moved there vector positions.

In [ ]:
initial_embeddings = embedding.weight.detach().cpu().clone()

We're going to save the random embeddings before, they will make no sense cause they are random, hence why we want to see if they move closer to each other.

# 2.Model Selection and Training (50 points) (SMALL MODEL)

In [39]:
class CNN_SIMPLE(nn.Module):

    def __init__(self, embedding, vocab_size, kernel_size=3):
        super().__init__()

        self.embedding = embedding #HERE WE ARE ATTCHING OUR EXTERNAL EMBEDDINGS TO THE MODEL SO THAT WHEN IT TRAINS THEY GET UPDATED

        self.conv = nn.Conv1d(200,128,kernel_size) #HERE WE HAVE 128 WEIGHTS OF 200 * 3, THAT MOVES ACORSS THE EMBEDDINGS

        self.fc = nn.Linear(128,vocab_size)

        self.kernel_size = kernel_size

    def forward(self, x):

        x = self.embedding(x)

        x = x.transpose(1, 2)

        x = F.pad(x, (self.kernel_size - 1, 0)) #HERE IS OUR CASUAL MASKING (WE DON'T WNAT TO SEE FUTURE EMBEDDINGS)

        x = torch.relu(self.conv(x)) #AND AN ACTIVATION FUNCTION

        x = x.transpose(1, 2)

        x = self.fc(x)

        return x

model_cnn_one = CNN_SIMPLE(embedding=embedding,vocab_size=len(vocab_tokens)).to(device) # WE ASSIGN OUR MODEL

criterion = nn.CrossEntropyLoss() #OK SO WE'RE GOING TO BE USING CROSS ENTROPY WHICH IS JUST A CLASSIFICATION LOSS TECHNIQUE

x = token_tensor[:, :-1]
y = token_tensor[:, 1:]

train_dataset = TensorDataset(x, y)

train_loader = DataLoader(train_dataset,batch_size=1,shuffle=True) #WE BATCH AS ONE OR TWO, EVEN WITH A GPU IT'S DIFFICULT TO HAVE BATCH OF 2

optimizer = torch.optim.Adam(model_cnn_one.parameters(), lr=0.001) #SO THE LEARNING RATE AND THEN THE OPTIMISER

epochs = 10 #I DON'T THINK WE'LL ACTUALLY TRAIN IT FOR 10 EPOCHS AS IT STILL TAKES AGES WITH 1

for epoch in range(epochs):

    model_cnn_one.train()

    for i, (inputs, labels) in enumerate(train_loader):

        optimizer.zero_grad()

        outputs = model_cnn_one(inputs)

        loss = criterion(outputs.reshape(-1, len(vocab_tokens)),labels.reshape(-1))

        loss.backward()

        optimizer.step()

        if torch.cuda.is_available():
            if (i + 1) % 100000 == 0:
                print(f"Epoch {epoch+1}, Batch {i+1}, Loss: {loss.item():.4f}")

        else:
            if (i + 1) % 50 == 0:
                print(f"Epoch {epoch+1}, Batch {i+1}, Loss: {loss.item():.4f}")
                #torch.save({"model_state_dict": model_cnn_one.state_dict(),"vocab_tokens": vocab_tokens,}, "/Users/benjaminbrooke/PycharmProjects/Agentic/MODEL")

KeyboardInterrupt: 

In [ ]:
from sklearn.decomposition import PCA
import plotly.express as px
import numpy as np

words = ["apple", "banana", "orange", "grape", "strawberry","pear", "peach", "mango", "pineapple", "watermelon", "cherry", "lemon", "lime", "kiwi", "blueberry", "raspberry", "plum", "coconut", "avocado", "tomato"]

ids = [vocab_tokens[word] for word in words] #WE TAKE OUR WORDS AND FIND IF THERE IS AN ID ATTCHED TO THE WORD.

before = initial_embeddings[ids].numpy() #HERE WE SEE FIND THE EMBEDDINGS BEFORE THE TRANING

after = model_cnn_one.embedding.weight.detach().cpu()[ids].numpy() #HERE WE SEE FIND THE EMBEDDINGS AFTER THE TRANING

pca = PCA(n_components=3) #OK SO WE CAN'T SEE A 200 DIMENTIONAL VECTOR SPACE, SO WE REDUCE TO 3

combined = pca.fit_transform(np.concatenate([before, after]))

fig = px.scatter_3d(x=combined[:, 0],y=combined[:, 1],z=combined[:, 2],text=words * 2,color=["Before"] * len(words) + ["After"] * len(words))

fig.show()

In [ ]:
torch.save({'model_state_dict': model_cnn_one.state_dict(),'vocab_tokens': vocab_tokens}, 'cnn_language_model.pth')

In [ ]:
def generate_text(model, prompt, vocab_tokens, max_new_tokens=50,
                  temperature=0.8, top_k=20):

    model.eval()

    device = next(model.parameters()).device

    id_to_word = {idx: word for word, idx in vocab_tokens.items()}

    words = prompt.lower().split()

    token_ids = [vocab_tokens.get(word, vocab_tokens['<UNK>'])for word in words]

    generated = torch.tensor([token_ids],dtype=torch.long,device=device)

    with torch.no_grad():

        for _ in range(max_new_tokens):
            context = generated[:, -500:]

            logits = model(context)

            next_token_logits = logits[:, -1, :]

            next_token_logits = next_token_logits / temperature

            values, indices = torch.topk(next_token_logits,top_k)

            probabilities = torch.softmax(values, dim=-1)

            sampled_index = torch.multinomial(probabilities,num_samples=1)

            next_token = indices.gather(-1,sampled_index)

            generated = torch.cat([generated, next_token],dim=1)

    output_words = [id_to_word[idx]for idx in generated[0].tolist()]

    return " ".join(output_words)

# 2.Model Selection and Training (50 points) (BIG MODEL)

In [ ]:
class BIGBlock(nn.Module):

    def __init__(self, channels, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()

        self.dilation = dilation
        self.kernel_size = kernel_size

        self.conv = nn.Conv1d(channels,channels * 2,kernel_size=kernel_size,dilation=dilation)

        self.residual_projection = nn.Conv1d(channels, channels, kernel_size=1)

        self.skip_projection = nn.Conv1d(channels, channels, kernel_size=1)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        residual = x

        padding = self.dilation * (self.kernel_size - 1)

        x = F.pad(x, (padding, 0))

        x = self.conv(x)

        tanh_part, sigmoid_part = x.chunk(2, dim=1)

        x = torch.tanh(tanh_part) * torch.sigmoid(sigmoid_part)

        x = self.dropout(x)

        residual = self.residual_projection(residual)

        skip = self.skip_projection(x)

        return residual + x, skip

In [ ]:
class CNN_LANGUAGE_MODEL(nn.Module):

    def __init__(self,embedding,channels=512,kernel_size=3,dilations=None,dropout=0.1):
        super().__init__()

        self.embedding = embedding

        self.input_projection = nn.Conv1d(embedding.embedding_dim,channels,kernel_size=1)

        if dilations is None:
            dilations = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512] * 2

        self.blocks = nn.ModuleList([
            BIGBlock(channels=channels,kernel_size=kernel_size,dilation=d,dropout=dropout)for d in dilations])

        self.output_projection = nn.Conv1d(channels,embedding.embedding_dim,kernel_size=1)

        self.output_bias = nn.Parameter(torch.zeros(embedding.num_embeddings))

    def forward(self, x):

        x = self.embedding(x)

        x = x.transpose(1, 2)

        x = self.input_projection(x)

        skip_connections = []

        for block in self.blocks:
            x, skip = block(x)
            skip_connections.append(skip)

        x = torch.stack(skip_connections).sum(dim=0)

        x = x / (len(self.blocks) ** 0.5)

        x = torch.relu(x)

        x = self.output_projection(x)

        x = x.transpose(1, 2)

        logits = F.linear(x,self.embedding.weight,self.output_bias)

        return logits

In [ ]:
embedding = nn.Embedding(num_embeddings=len(vocab_tokens),embedding_dim=512).to(device)

model = CNN_LANGUAGE_MODEL(embedding=embedding,channels=512,kernel_size=3,dropout=0.1).to(device)

x = token_tensor[:, :-1]
y = token_tensor[:, 1:]

train_dataset = TensorDataset(x, y)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(),lr=0.003,weight_decay=0.01)

In [ ]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for inputs, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs.reshape(-1, len(vocab_tokens)),labels.reshape(-1))

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, "f"Average Loss: {total_loss / len(train_loader)}")

In [ ]:
torch.save({'model_state_dict': model,'vocab_tokens': vocab_tokens}, 'LARGE_cnn.pth')

# 3.Test the Model

In [ ]:
def generate_text(model, prompt, vocab_tokens, max_new_tokens=50,
                  temperature=0.8, top_k=20):

    model.eval()

    device = next(model.parameters()).device

    id_to_word = {idx: word for word, idx in vocab_tokens.items()}

    words = prompt.lower().split()

    token_ids = [vocab_tokens.get(word, vocab_tokens['<UNK>'])for word in words]

    generated = torch.tensor([token_ids],dtype=torch.long,device=device)

    with torch.no_grad():

        for _ in range(max_new_tokens):
            context = generated[:, -500:]

            logits = model(context)

            next_token_logits = logits[:, -1, :]

            next_token_logits = next_token_logits / temperature

            values, indices = torch.topk(next_token_logits,top_k)

            probabilities = torch.softmax(values, dim=-1)

            sampled_index = torch.multinomial(probabilities,num_samples=1)

            next_token = indices.gather(-1,sampled_index)

            generated = torch.cat([generated, next_token],dim=1)

    output_words = [id_to_word[idx]for idx in generated[0].tolist()]

    return " ".join(output_words)